In [ ]:
# llamado de librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Importar librerias de Keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPooling2D
# Importar libreria de preprocesamiento
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Generar valores o datos aleatorios
np.random.seed(123)
# Crear serie de tiempo
time_series = np.sin(np.arange(0,100,0.1)) + np.random.normal(0,0.1,1000)

# variable externa
external_variable = np.random.random(1000)

# Generar mi variable de tiempo
time = np.arange(0,100,0.1)

In [ ]:
# Crear mi dataset
df = pd.DataFrame({'Time':time,
                   'TimeSeries':time_series,
                   'ExternalVariable':external_variable})
df.head(2)

In [ ]:
# Escalar los valores
scaler = MinMaxScaler(feature_range = (0,1))
df_scaler = scaler.fit_transform(df[['TimeSeries','ExternalVariable']])

In [ ]:
df_scaler

In [ ]:
# Definimos la funcion de la arquitectura X - y.
def create_dataset(data, time_steps=1):
    Xs, ys = [], []
    for i in range(len(data) - time_steps):
        #v = X.iloc[i:(i + time_steps)].values
        Xs.append(data[i:(i + time_steps),:])
        ys.append(data[i + time_steps,0])
    return np.array(Xs), np.array(ys)

In [ ]:
X,y = create_dataset(df_scaler,time_steps = 2)

In [ ]:
X

In [ ]:
split = 700
X_train, y_train, X_test, y_test = X[:split], y[:split], X[split:], y[split:]

In [ ]:
# Importando de keras las librerias mas importantes!
from keras.models import Sequential # Arquitectura de red neuronal!
from keras.layers import Dense      # Capa densa!
from keras.layers import LSTM       # Capa recurrente
from keras.layers import Dropout    # Evita el overfitting (Inactiva algunas neuronas)

def lstm_architecture(X_data,rate_dropout):
    # Inicializando the RNN
    model = Sequential()

    # 1ra capa LSTM y Dropout para regularización.
    # input_shape (amplitude,1)
    model.add(LSTM(units = 250, return_sequences = True, input_shape=(X_data.shape[1], X_data.shape[2])))
    # 20% de las neuronas serán ignoradas durante el training (20%xNodos = 10)
    # Para hacer menos probable el overfiting
    model.add(Dropout(rate=rate_dropout))

    # 2da capa LSTM y Dropout para regularización.
    model.add(LSTM(units = 250, return_sequences = True))
    model.add(Dropout(rate=rate_dropout))

    # 3ra capa LSTM y Dropout para regularización.
    model.add(LSTM(units = 250, return_sequences = True))
    model.add(Dropout(rate=rate_dropout))

    # 4ta capa LSTM y Dropout para regularización.
    model.add(LSTM(units = 250, return_sequences = False))
    model.add(Dropout(rate=rate_dropout))

    # Capa de Salida!
    model.add(Dense(units = 1))

    # Resumen del modelo!
    model.summary()

    return model

In [ ]:
import datetime
print('Iniciando a las: ', datetime.datetime.now())
print("...")

# Compiling the RNN
model_1 = lstm_architecture(X_data=X_train,rate_dropout=0.2)
model_1.compile(optimizer = 'adam', loss = 'mean_squared_error')

In [ ]:
import datetime
print('Iniciando a las: ', datetime.datetime.now())
print("...")
# Ejecutamos la RNN!

history = model_1.fit(X_train,
                    y_train,
                    epochs=50,
                    validation_data=(X_test,y_test),
                    batch_size=32,
                    shuffle=False)

print("...")
print('Terminando a las: ', datetime.datetime.now())

In [ ]:
# Revisamos algunos parametros de ajuste del modelo!
plt.plot(history.history['loss'], label='train')
plt.legend();
plt.show()

In [ ]:
# Predecimos sobre la data de test!
y_pred = model_1.predict(X_test)

In [ ]:
# Visualizamos los resultados!
plt.figure(num=None, figsize=(15, 6), dpi=80, facecolor='w', edgecolor='k')
plt.plot(np.arange(len(y_train), len(y_train) + len(y_test)), y_test.flatten(), marker='.', label="true")
plt.plot(np.arange(len(y_train), len(y_train) + len(y_test)), y_pred.flatten(), 'r', marker='.', label="prediction")
plt.plot(np.arange(0, len(y_train)), y_train.flatten(), 'g', marker='.', label="history")
plt.ylabel('Count')
plt.xlabel('Time Step')
plt.legend()
plt.show()

In [ ]:
df['TimeSeries'].describe()

In [ ]:
train_max = 1.232726
train_min = -1.208263

In [ ]:
# Regresamos la informacion a los valores originales!
y_test = y_test*(train_max - train_min) + train_min
y_pred = y_pred*(train_max - train_min) + train_min
y_train = y_train*(train_max - train_min) + train_min

In [ ]:
# Vemos algunos indicadores del ajuste!
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f'RMSE: ',rmse)

# Definimos y calculamos el MAPE (mean_absolute_percentage_error)
y_test, y_pred = np.array(y_test), np.array(y_pred)
print(f'MAPE: ',np.mean(np.abs((y_test - y_pred) / y_test)) * 100)